In [28]:
# Download PDF file
import os
import requests

# Get PDF document
pdf_path = "human-nutrition-text.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  # The URL of the PDF you want to download
  url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

  # The local filename to save the downloaded file
  filename = pdf_path

  # Send a GET request to the URL
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Open a file in binary write mode and save the content to it
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File human-nutrition-text.pdf exists.


In [29]:
# Requires !pip install PyMuPDF, see: https://github.com/pymupdf/pymupdf
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

def open_and_read_pdf(pdf_path: str)-> list[dict]:
    """
        Opens a PDF file, reads its text content page by page, and collects statistics.

        Parameters:
            pdf_path (str): The file path to the PDF document to be opened and read.

        Returns:
            list[dict]: A list of dictionaries, each containing the page number
            (adjusted), character count, word count, sentence count, token count, and the extracted text
            for each page.
    """
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
            text = page.get_text()  # get plain text encoded as UTF-8
            text = text_formatter(text)
            pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                    "page_char_count": len(text),
                                    "page_word_count": len(text.split(" ")),
                                    "page_sentence_count_raw": len(text.split(". ")),
                                    "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                    "text": text})
    return pages_and_texts
    
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]



1208it [00:01, 858.42it/s]


[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

# Chunking Strategy
## 1: Fixed Chunking

In [30]:
def chunk_text(text: str, chunk_size: int = 500) -> list:
    # >> Input: one page of text. Output: a list of strings (the chunks).
    # >> chunk_size=500 is a DEFAULT: used if you don't pass a size.
    """
    Splits text into chunks of approx. `chunk_size` characters.
    """
    chunks = []              # >> finished chunks collect here
    current_chunk = ''       # >> the chunk being filled right now (starts empty)
    words = text.split()     # >> list of words; .split() with no argument splits on any whitespace

    for word in words:       # >> look at one word at a time
        # Check if adding the word exceeds chunk size
        if len(current_chunk) + len(word) + 1 <= chunk_size:
            # >> current length + this word + 1 space. Still within 500? Then it fits.
            current_chunk += (word + ' ')
            # >> += adds to the end of the string: "Human " + "Nutrition " -> "Human Nutrition "
        else:
            # Store current chunk and start new one
            chunks.append(current_chunk.strip())
            # >> it doesn't fit: save the full chunk (.strip() removes the trailing space)
            current_chunk = word + ' '
            # >> the new chunk STARTS with the word that didn't fit, so no word is lost or cut

    # Add the last chunk if not empty
    if current_chunk:
        # >> after the loop, the last chunk is usually partly full. An empty string
        # >> counts as False, so this saves it only if it holds something.
        chunks.append(current_chunk.strip())

    return chunks

In [31]:
def chunk_pdf_pages(pages_and_texts: list, chunk_size: int = 500) -> list[dict]:
    """
    Takes PDF pages with text and splits them into chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []                         # >> every chunk from every page ends up here

    for page in pages_and_texts:            # >> outer loop: one page dictionary at a time
        page_number = page["page_number"]   # >> read values from the page dictionary by key
        page_text = page["text"]

        chunks = chunk_text(page_text, chunk_size=chunk_size)
        # >> call Function 1 on this page only. Chunks never cross from one page into the next.
        # >> chunk_size=chunk_size is a KEYWORD ARGUMENT: parameter name = your value.

        for i, chunk in enumerate(chunks):
            # >> inner loop over this page's chunks. enumerate gives a counter i (0, 1, 2...)
            # >> so we know each chunk's position on its page.
            all_chunks.append({
                "page_number": page_number,          # >> which page: needed later for citations
                "chunk_index": i,                    # >> 0 = first chunk on the page, 1 = second...
                "chunk_char_count": len(chunk),      # >> characters
                "chunk_word_count": len(chunk.split()),  # >> words
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk                  # >> the text itself
            })
    return all_chunks

In [32]:
chunked_pages = chunk_pdf_pages(pages_and_texts, chunk_size=500)
print(f"Total chunks: {len(chunked_pages)}")
print(f"First chunk (page {chunked_pages[0]['page_number']}): {chunked_pages[0]['chunk_text'][:200]}...")

Total chunks: 3321
First chunk (page -41): Human Nutrition: 2020 Edition...


## visualize chunks

In [33]:
import random, textwrap
# >> two standard-library modules imported on one line (comma-separated)

def _scattered_indices(n: int, k: int, jitter_frac: float = 0.08) -> list[int]:
    # >> A leading underscore (_name) is a convention meaning "helper, used inside this notebook only".
    # >> n = how many chunks exist (3321); k = how many to pick (5).
    """Evenly spaced anchors + random jitter → indices scattered across [0, n-1]."""
    if k <= 0:
        return []
    if k == 1:
        return [random.randrange(n)]        # >> one random number from 0 to n-1
    anchors = [int(round(i * (n - 1) / (k - 1))) for i in range(k)]
    # >> LIST COMPREHENSION: k evenly spaced positions.
    # >> For n=3321, k=5: [0, 830, 1660, 2490, 3320] (start, quarter, middle, three-quarters, end)
    out, seen = [], set()
    # >> out = the chosen positions; seen = a SET (no duplicates, fast "is it in here?" check)
    radius = max(1, int(n * jitter_frac))
    # >> 8% of 3321 is about 265: how far each pick may wander from its anchor
    for a in anchors:
        lo, hi = max(0, a - radius), min(n - 1, a + radius)
        # >> the window around the anchor, kept inside 0..n-1 by max/min
        j = random.randint(lo, hi)          # >> random position in that window (both ends included)
        if j not in seen:
            out.append(j); seen.add(j)      # >> ";" puts two statements on one line
    while len(out) < k:
        # >> WHILE LOOP: repeat until we have k picks (only needed if there were duplicates)
        r = random.randrange(n)
        if r not in seen:
            out.append(r); seen.add(r)
    return out

In [34]:
def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    # >> c = one chunk dictionary. Returns ONE multi-line string (the whole box).
    header = (
        f" Chunk p{c['page_number']} · idx {c['chunk_index']}  |  "
        f"chars {c['chunk_char_count']} · words {c['chunk_word_count']} · ~tokens {c['chunk_token_count']} "
    )
    # >> Two f-strings side by side inside brackets are joined automatically into one string.
    # Wrap body text, avoid breaking long words awkwardly
    wrapped_lines = textwrap.wrap(
        c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False
    )
    # >> textwrap.wrap cuts the text into a list of lines of at most 96 characters,
    # >> without cutting through a word.
    content_width = max([0, *map(len, wrapped_lines)])
    # >> map(len, lines) gives the length of each line; * UNPACKS them into the list.
    # >> The 0 is a safety value so max() still works if there are no lines.
    # >> Result: the length of the longest line.
    box_width = max(len(header), content_width + 2)  # +2 for side padding
    # >> the box must fit whichever is wider: the header or the longest line

    top    = "╔" + "═" * box_width + "╗"          # >> "═" * n repeats the character n times
    hline  = "║" + header.ljust(box_width) + "║"  # >> .ljust(n) pads with spaces on the right to width n
    sep    = "╟" + "─" * box_width + "╢"
    body   = "\n".join("║ " + line.ljust(box_width - 2) + " ║" for line in wrapped_lines) or \
             ("║ " + "".ljust(box_width - 2) + " ║")
    # >> "\n".join(...) glues the lines together with newlines between them.
    # >> The part inside join(...) is a GENERATOR EXPRESSION, like a list comprehension without [ ].
    # >> "A or B" returns B when A is empty: if there were no lines, draw one blank line instead.
    # >> The "\" at the end of a line continues the statement on the next line.
    bottom = "╚" + "═" * box_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])

In [35]:
def show_random_chunks(pages_and_texts: list, chunk_size: int = 500, k: int = 5, seed: int | None = 42):
    # >> "int | None" means: either a number or None (nothing).
    if seed is not None:
        random.seed(seed)
        # >> A fixed SEED makes "random" repeatable: seed=42 gives the same 5 chunks every run.
        # >> Pass seed=None to get different chunks each time.
    all_chunks = chunk_pdf_pages(pages_and_texts, chunk_size=chunk_size)
    # >> Note: this chunks the whole book AGAIN instead of reusing chunked_pages. It's fast here,
    # >> so it doesn't matter, but it means you can try a different chunk_size directly in this call.
    if not all_chunks:
        print("No chunks to display.")
        return
    idxs = _scattered_indices(len(all_chunks), k)
    print(f"Showing {len(idxs)} scattered random chunks out of {len(all_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        # >> enumerate(..., 1) starts the counter at 1 instead of 0, for "#1, #2 ..."
        print(f"#{i}")
        print(_draw_boxed_chunk(all_chunks[idx]))
        print()                              # >> blank line between boxes

# ---------- Run ----------
assert 'pages_and_texts' in globals(), "Run: pages_and_texts = open_and_read_pdf(pdf_path) first."
# >> globals() is a dictionary of every variable defined in the notebook so far.
# >> ASSERT stops with that message if the condition is False: here, if Step 2 wasn't run.
show_random_chunks(pages_and_texts, chunk_size=500, k=5, seed=42)

Showing 5 scattered random chunks out of 3321 total:

#1
╔══════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Chunk p-9 · idx 0  |  chars 290 · words 49 · ~tokens 72.5                                        ║
╟──────────────────────────────────────────────────────────────────────────────────────────────────╢
║ Skylar Hara Skylar Hara is an undergraduate student student in the Tropical Agriculture and the  ║
║ Environment program at the University of Hawai‘i at Mānoa. She has a growing love for plants and ║
║ hopes to go to graduate school to conduct research in the future. About the Contributors |       ║
║ xxxiii                                                                                           ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝

#2
╔═════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Chunk p198 · idx 0  |  chars 

# Chunking Strategy: 2 Semantic Chunking

In [36]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt', quiet=True)
# >> nltk's sentence splitter needs a data file, downloaded once to your computer.

# Load once globally
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
# >> Downloads a small embedding model (about 90 MB) the first time, then reuses it from a cache.
# >> It turns any sentence into a list of 384 numbers that represent its meaning.
# >> "Load once globally": loading it inside the function would reload it for every page, which is very slow.

def semantic_chunk_text(text: str, similarity_threshold: float = 0.8, max_tokens: int = 500) -> list:
    """
    Splits text into semantic chunks based on sentence similarity and max token length.
    """
    sentences = nltk.sent_tokenize(text)
    # >> Splits the page into sentences. Smarter than splitting on ". " because it
    # >> handles cases like "Dr. Smith" and "e.g." correctly.
    if not sentences:
        return []            # >> blank page: no chunks

    embeddings = semantic_model.encode(sentences)
    # >> One call embeds ALL sentences on the page at once (faster than one by one).
    # >> Result: a NumPy array of shape (number_of_sentences, 384); row i is sentence i.

    chunks = []
    current_chunk = [sentences[0]]       # >> the chunk is a LIST of sentences, joined into text later
    current_embedding = embeddings[0]    # >> the "meaning" of the chunk so far: starts as sentence 0

    for i in range(1, len(sentences)):
        # >> range(1, n) counts 1, 2, ..., n-1. Sentence 0 is already in the chunk.
        sim = cosine_similarity([current_embedding], [embeddings[i]])[0][0]
        # >> Cosine similarity: how close two meanings are. 1 = same meaning, 0 = unrelated.
        # >> sklearn's function compares LISTS of vectors and returns a TABLE (matrix) of scores,
        # >> so each vector is wrapped in [ ], and [0][0] takes the single score out of the 1x1 table.
        chunk_token_count = len(" ".join(current_chunk)) // 4
        # >> Size of the chunk so far, in approximate tokens.
        # >> "//" is INTEGER DIVISION: 1003 // 4 = 250 (drops the remainder).

        if sim >= similarity_threshold and chunk_token_count < max_tokens:
            # >> "and": BOTH must be true: similar enough AND the chunk isn't too big yet.
            current_chunk.append(sentences[i])
            current_embedding = np.mean([current_embedding, embeddings[i]], axis=0)
            # >> Update the chunk's meaning: average the old meaning with the new sentence.
            # >> axis=0 averages position by position, giving another 384-number vector.
        else:
            chunks.append(" ".join(current_chunk))   # >> close the chunk: join its sentences with spaces
            current_chunk = [sentences[i]]           # >> the new chunk starts with this sentence
            current_embedding = embeddings[i]

    if current_chunk:
        chunks.append(" ".join(current_chunk))       # >> save the last chunk

    return chunks


from tqdm.auto import tqdm

def semantic_chunk_pdf_pages(pages_and_texts: list,
                             similarity_threshold: float = 0.8,
                             max_tokens: int = 500) -> list[dict]:
    """
    Takes PDF pages with text and splits them into semantic chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Semantic chunking pages"):
        # >> Same pattern as chunk_pdf_pages in 3.1. tqdm adds a progress bar;
        # >> desc= is the label shown next to it.
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = semantic_chunk_text(page_text,
                                     similarity_threshold=similarity_threshold,
                                     max_tokens=max_tokens)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })
    return all_chunks

In [37]:
import nltk
nltk.download('punkt_tab')
# >> Newer nltk versions need this second data file too. Without it you get a LookupError.
semantic_chunked_pages = semantic_chunk_pdf_pages(pages_and_texts,
                                                  similarity_threshold=0.75,
                                                  max_tokens=500)
# >> He overrides the default 0.8 with 0.75 here.

print(f"Total semantic chunks: {len(semantic_chunked_pages)}")
print(f"First semantic chunk (page {semantic_chunked_pages[0]['page_number']}):")
print(semantic_chunked_pages[0]['chunk_text'][:200] + "...")
# >> "+" joins two strings: the first 200 characters, then "..."

[nltk_data] Downloading package punkt_tab to C:\Users\Parvez
[nltk_data]     Alam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Semantic chunking pages: 100%|██████████| 1208/1208 [02:12<00:00,  9.08it/s]

Total semantic chunks: 12092
First semantic chunk (page -41):
Human Nutrition: 2020 Edition...


In [38]:
def show_random_semantic_chunks(semantic_chunked_pages: list, k: int = 5, seed: int | None = 42):
    if seed is not None:
        random.seed(seed)
    idxs = _scattered_indices(len(semantic_chunked_pages), k)
    print(f"Showing {len(idxs)} scattered random SEMANTIC chunks out of {len(semantic_chunked_pages)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(semantic_chunked_pages[idx]))
        print()

assert 'semantic_chunked_pages' in globals() and len(semantic_chunked_pages) > 0, \
    "Run your semantic chunking code first to define `semantic_chunked_pages`."
show_random_semantic_chunks(semantic_chunked_pages, k=5, seed=42)

Showing 5 scattered random SEMANTIC chunks out of 12092 total:

#1
╔═════════════════════════════════════════════════════════╗
║ Chunk p53 · idx 1  |  chars 27 · words 5 · ~tokens 6.75 ║
╟─────────────────────────────────────────────────────────╢
║ THE HUMAN  BODY  Chapter 2.                             ║
╚═════════════════════════════════════════════════════════╝

#2
╔══════════════════════════════════════════════════════════╗
║ Chunk p231 · idx 9  |  chars 56 · words 9 · ~tokens 14.0 ║
╟──────────────────────────────────────────────────────────╢
║ The simplest unit of a carbohydrate is a monosaccharide. ║
╚══════════════════════════════════════════════════════════╝

#3
╔════════════════════════════════════════════════════════╗
║ Chunk p508 · idx 5  |  chars 8 · words 2 · ~tokens 2.0 ║
╟────────────────────────────────────────────────────────╢
║ Hill JO.                                               ║
╚════════════════════════════════════════════════════════╝

#4
╔════════════════════

In [39]:
import nltk
from tqdm.auto import tqdm
nltk.download("punkt")

def recursive_chunk_text(text: str, max_chunk_size: int = 1000, min_chunk_size: int = 100) -> list:
    """
    Recursively splits a block of text into chunks that fit within size constraints.
    Tries splitting by sections, then newlines, then sentences.
    """
    def split_chunk(chunk: str) -> list:
        # >> A NESTED FUNCTION: defined inside another function, usable only in there.
        # >> It can read max_chunk_size from the outer function without being passed it
        # >> (this is called a CLOSURE).

        #  Base case
        if len(chunk) <= max_chunk_size:
            return [chunk]
            # >> BASE CASE: the stopping rule of recursion. Small enough, so keep it whole.
            # >> Every recursive function needs one, or it would call itself forever.

        #  Try splitting by double newlines
        sections = chunk.split("\n\n")
        # >> "\n\n" = a blank line, which usually separates sections
        if len(sections) > 1:
            # >> more than 1 piece means a blank line was actually found
            result = []
            for section in sections:
                if section.strip():
                    # >> skip pieces that are only spaces (an empty string counts as False)
                    result.extend(split_chunk(section.strip()))
                    # >> RECURSION: split_chunk calls ITSELF on the smaller piece.
                    # >> If the piece is small, the base case returns it as is;
                    # >> if not, it gets split further.
                    # >> .extend() adds each item of a list; .append() would add the whole list as ONE item:
                    # >>   [1].extend([2, 3]) -> [1, 2, 3]      [1].append([2, 3]) -> [1, [2, 3]]
            return result

        # Try splitting by single newline
        sections = chunk.split("\n")
        # >> Reached only if there was no blank line. "\n" usually separates paragraphs or lines.
        if len(sections) > 1:
            result = []
            for section in sections:
                if section.strip():
                    result.extend(split_chunk(section.strip()))
            return result

        # Fallback: split by sentences
        # >> No newlines at all: pack whole sentences into chunks of at most max_chunk_size
        sentences = nltk.sent_tokenize(chunk)
        chunks, current_chunk, current_size = [], [], 0
        # >> three variables set on one line

        for sentence in sentences:
            if current_size + len(sentence) > max_chunk_size:
                # >> adding this sentence would overflow
                if current_chunk:
                    chunks.append(" ".join(current_chunk))   # >> save the chunk so far
                current_chunk = [sentence]                   # >> start a new chunk with this sentence
                current_size = len(sentence)
            else:
                current_chunk.append(sentence)               # >> fits: add it
                current_size += len(sentence)

        if current_chunk:
            chunks.append(" ".join(current_chunk))           # >> save the last one

        return chunks

    return split_chunk(text)
    # >> The outer function only starts the recursion on the whole text.


def recursive_chunk_pdf_pages(pages_and_texts: list,
                              max_chunk_size: int = 1000,
                              min_chunk_size: int = 100) -> list[dict]:
    # >> Same page-loop pattern as 3.1 and 3.2: only the inner function changes.
    """
    Takes PDF pages with text and splits them into recursive chunks.

    Returns a list of dicts with page_number, chunk_index, and chunk_text.
    """
    all_chunks = []

    for page in tqdm(pages_and_texts, desc="Recursive chunking pages"):
        page_number = page["page_number"]
        page_text = page["text"]

        chunks = recursive_chunk_text(page_text,
                                      max_chunk_size=max_chunk_size,
                                      min_chunk_size=min_chunk_size)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page_number": page_number,
                "chunk_index": i,
                "chunk_char_count": len(chunk),
                "chunk_word_count": len(chunk.split()),
                "chunk_token_count": len(chunk) / 4,  # rough token estimate
                "chunk_text": chunk
            })
    return all_chunks

[nltk_data] Downloading package punkt to C:\Users\Parvez
[nltk_data]     Alam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Parvez
[nltk_data]     Alam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [40]:
recursive_chunked_pages = recursive_chunk_pdf_pages(pages_and_texts,
                                                    max_chunk_size=800,
                                                    min_chunk_size=100)

print(f"Total recursive chunks: {len(recursive_chunked_pages)}")
print(f"First recursive chunk (page {recursive_chunked_pages[0]['page_number']}):")
print(recursive_chunked_pages[0]['chunk_text'][:200] + "...")

Recursive chunking pages: 100%|██████████| 1208/1208 [00:00<00:00, 2491.67it/s]

Total recursive chunks: 2433
First recursive chunk (page -41):
Human Nutrition: 2020 Edition...


In [41]:
print(sum("\n" in p["text"] for p in pages_and_texts))   # >> counts pages containing a newline

0


In [42]:
for page_index, page in enumerate(fitz.open(pdf_path)):
    pages_and_texts[page_index]["raw_text"] = page.get_text()   # >> text with newlines kept

# Step 3.4: Structural chunking (video: Part 1, 2:57:42)

In [43]:
# We detect a new CHAPTER whenever a page contains "University of Hawaii" header.
# Each chapter = pages from one header until the page before the next header.

import re
# >> "re" = REGULAR EXPRESSIONS: a mini-language for finding text patterns. Standard library.
import random
import textwrap

# 1) Helper to detect "chapter start" pages
def _is_chapter_header_page(text: str) -> bool:
    # Robust to punctuation/diacritics differences; matches the recurring header
    # e.g., "UNIVERSITY OF HAWAI'I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM"
    return re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE) is not None
    # >> r"..." is a RAW STRING: backslashes stay as typed, so "\s" reaches the regex intact.
    # >> \s+ means "one or more whitespace characters", so "of  hawai" (two spaces) still matches.
    # >> "hawai" stops before the apostrophe, so HAWAI'I, HAWAII and Hawaiʻi all match.
    # >> re.IGNORECASE: capitals don't matter.
    # >> re.search returns a MATCH object if found, or None if not.
    # >> "... is not None" turns that into True/False, which is why the function returns bool.

def _guess_title_from_page(text: str) -> str:
    """
    Best-effort chapter title guess = the text before the 'University of Hawai' header line.
    Falls back to the first ~120 characters.
    """
    m = re.search(r"university\s+of\s+hawai", text, flags=re.IGNORECASE)
    if m:
        # >> a Match object counts as True; None counts as False
        title = text[:m.start()].strip()
        # >> m.start() = the character position where the match begins.
        # >> text[:position] = everything BEFORE it, which is where the chapter title sits.
        # keep it readable
        title = re.sub(r"\s+", " ", title).strip()
        # >> re.sub = find and replace with a pattern: any run of whitespace becomes one space
        if 10 <= len(title) <= 180:
            # >> CHAINED COMPARISON: means 10 <= len(title) AND len(title) <= 180
            return title
    # fallback
    t = re.sub(r"\s+", " ", text).strip()
    return t[:120] if t else "Untitled Chapter"
    # >> CONDITIONAL EXPRESSION: value_if_true if condition else value_if_false

# 2) Build chapter chunks
def chapter_chunk_pdf_pages(pages_and_texts: list[dict]) -> list[dict]:
    """
    Returns a list of chapter chunks:
    [
      {
        'chapter_index': int,
        'title': str,
        'page_start': int,   # adjusted page number (your -41 offset)
        'page_end': int,
        'chunk_char_count': int,
        'chunk_word_count': int,
        'chunk_token_count': float,   # ~chars/4
        'chunk_text': str
      }, ...
    ]
    """
    # >> This docstring documents the OUTPUT SHAPE: a good habit for functions returning dictionaries.
    if not pages_and_texts:
        return []

    # Find all page indices that look like the start of a chapter
    chapter_starts = []
    for i, p in enumerate(pages_and_texts):
        txt = p["text"]
        if _is_chapter_header_page(txt):
            chapter_starts.append(i)
    # >> Note: these are LIST POSITIONS (0..1207), not book page numbers.
    # >> e.g. chapter_starts = [2, 64, 71, ...]

    # If nothing detected, return empty (or treat entire doc as one chunk)
    if not chapter_starts:
        # Treat entire doc as one "chapter"
        all_text = " ".join(p["text"] for p in pages_and_texts).strip()
        # >> GENERATOR EXPRESSION inside join: every page's text, joined with spaces
        return [{
            "chapter_index": 0,
            "title": _guess_title_from_page(pages_and_texts[0]["text"]),
            "page_start": pages_and_texts[0]["page_number"],
            "page_end": pages_and_texts[-1]["page_number"],   # >> [-1] = the LAST item
            "chunk_char_count": len(all_text),
            "chunk_word_count": len(all_text.split()),
            "chunk_token_count": round(len(all_text) / 4, 2),
            "chunk_text": all_text
        }]
        # >> A safety net: if the pattern never matches (e.g. a different PDF), you still get one chunk.

    # Build chapter ranges (start -> next_start-1)
    chapter_chunks = []
    for ci, s in enumerate(chapter_starts):
        # >> ci = chapter number (0, 1, 2...); s = the page position where it starts
        e = (chapter_starts[ci + 1] - 1) if (ci + 1 < len(chapter_starts)) else (len(pages_and_texts) - 1)
        # >> END page: one before the NEXT chapter's start, or the last page for the final chapter.
        if e < s:
            continue  # guard (shouldn't happen)
            # >> "continue" skips the rest of this loop pass and moves to the next chapter

        pages = pages_and_texts[s:e + 1]
        # >> SLICING excludes the end, so e + 1 is needed to INCLUDE page e
        text_concat = " ".join(p["text"] for p in pages).strip()   # >> the whole chapter as one string
        title = _guess_title_from_page(pages[0]["text"])

        chapter_chunks.append({
            "chapter_index": ci,
            "title": title,
            "page_start": pages[0]["page_number"],
            "page_end": pages[-1]["page_number"],
            "chunk_char_count": len(text_concat),
            "chunk_word_count": len(text_concat.split()),
            "chunk_token_count": round(len(text_concat) / 4, 2),
            # >> round(x, 2) keeps 2 decimal places
            "chunk_text": text_concat
        })

    return chapter_chunks

In [44]:
structure_chunked_pages = chapter_chunk_pdf_pages(pages_and_texts)

print(f"Total chapter-based chunks: {len(structure_chunked_pages)}")
if structure_chunked_pages:
    first = structure_chunked_pages[0]
    print(f"First chapter (pages {first['page_start']}–{first['page_end']}): {first['title']}")
    print(first['chunk_text'][:200] + "...")
else:
    print("No chapters detected.")

Total chapter-based chunks: 171
First chapter (pages -39–-39): Human Nutrition: 2020 Edition
Human Nutrition: 2020  Edition  UNIVERSITY OF HAWAI‘I AT MĀNOA  FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM  ALAN TITCHENAL, SKYLAR HARA,  NOEMI ARCEO CAACBAY, WILLIAM  MEINKE-LAU, YA-YUN YANG, MARIE  K...


In [45]:
# 3) (Optional) Pretty print a few chapter chunks to inspect
def _draw_boxed_chunk(c: dict, wrap_at: int = 96) -> str:
    # >> Same name as the box function in 3.1 to 3.3, but REDEFINED here for chapters.
    # >> Defining a function again with the same name REPLACES the old one from this point on.
    # >> (That's why Steps 3.1 to 3.3 repeat their own copy: each cell uses the one defined in it.)
    header = (
        f" Chapter {c['chapter_index']}  |  {c['title'][:120]}  "
        f"| p{c['page_start']}–{c['page_end']}  |  ~tokens {c['chunk_token_count']}"
    )
    # >> Chapter header: number, title (cut to 120 characters), page range, token estimate.
    # >> Chapter chunks have page_start/page_end instead of page_number/chunk_index,
    # >> so the 3.1 box function would fail here with a KeyError. That's why this one is different.
    wrapped = textwrap.wrap(c["chunk_text"], width=wrap_at, break_long_words=False, replace_whitespace=False)
    # >> chapter text split into lines of at most 96 characters, without cutting words
    content_width = max(len(header), *(len(x) for x in wrapped)) if wrapped else len(header)
    # >> Read it in two parts:
    # >>   max(len(header), *(len(x) for x in wrapped))
    # >>       the * UNPACKS every line length into max(), like max(80, 96, 91, 95, ...)
    # >>       so the result is the widest of the header and all the lines.
    # >>   ... if wrapped else len(header)
    # >>       CONDITIONAL EXPRESSION: if there are no lines (empty chapter), just use the header width.
    top    = "╔" + "═"*content_width + "╗"
    hline  = "║" + header.ljust(content_width) + "║"
    sep    = "╟" + "─"*content_width + "╢"
    body   = "\n".join("║ " + line.ljust(content_width-2) + " ║" for line in wrapped[:12]) or \
             ("║ " + "".ljust(content_width-2) + " ║")
    # >> wrapped[:12] = ONLY THE FIRST 12 LINES. A chapter can have hundreds of lines,
    # >> so this shows a preview, not the whole chapter.
    # >> content_width-2 leaves room for the "║ " and " ║" borders, so every row has the same width.
    # >> "... or \" = if there were no lines, draw one empty row instead.
    bottom = "╚" + "═"*content_width + "╝"
    return "\n".join([top, hline, sep, body, bottom])
    # >> five pieces joined with newlines = the complete box as one string

def show_random_chapter_chunks(chapter_chunks: list[dict], k: int = 5, seed: int | None = 42):
    if not chapter_chunks:
        print("No chapter chunks to display."); return
        # >> Two statements on one line, separated by ";". "return" with no value just exits.
        # >> An empty list counts as False, so "not chapter_chunks" means "the list is empty".
    if seed is not None:
        random.seed(seed)
        # >> repeatable picks; the run below uses seed=21, not the default 42
    k = min(k, len(chapter_chunks))
    # >> never ask for more chapters than exist (random.sample would raise an error)
    idxs = random.sample(range(len(chapter_chunks)), k)
    # >> range(n) = the numbers 0..n-1; random.sample picks k DIFFERENT ones.
    # >> Plain random here, not the evenly spread _scattered_indices used in 3.1 to 3.3.
    print(f"Showing {k} random chapters out of {len(chapter_chunks)} total:\n")
    for i, idx in enumerate(idxs, 1):
        print(f"#{i}")
        print(_draw_boxed_chunk(chapter_chunks[idx]))
        print()

# 4) Run
assert 'pages_and_texts' in globals(), "Run your base PDF loader first to define `pages_and_texts`."
# >> stops with this message if Step 2 hasn't been run
chapter_chunks = chapter_chunk_pdf_pages(pages_and_texts)
# >> builds the chapters again under a new name. Same data as structure_chunked_pages from Cell 2.
# >> Step 4 uses structure_chunked_pages, so the extra name does no harm.
print(f"Total chapters detected: {len(chapter_chunks)}")
if chapter_chunks:
    print(f"First chapter: {chapter_chunks[0]['title']}  (p{chapter_chunks[0]['page_start']}–{chapter_chunks[0]['page_end']})")

# Inspect a few
show_random_chapter_chunks(chapter_chunks, k=5, seed=21)

Total chapters detected: 171
First chapter: Human Nutrition: 2020 Edition  (p-39–-39)
Showing 5 random chapters out of 171 total:

#1
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Chapter 42  |  Image by John Towner on unsplash.co m / CC0 Lifestyles and Nutrition  | p21–29  |  ~tokens 3135.25║
╟──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╢
║ Image by  John Towner  on  unsplash.co m / CC0  Lifestyles and Nutrition  UNIVERSITY OF HAWAI‘I                  ║
║ AT MĀNOA FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM AND HUMAN NUTRITION PROGRAM  In addition to                   ║
║ nutrition, health is affected by genetics, the  environment, life cycle, and lifestyle. One                      ║
║ facet of lifestyle is your  dietary habits. Recall that we discussed briefly how nutrition                       ║
║ affects  health. A greater discussion of this

In [47]:
# 4) Run
assert 'pages_and_texts' in globals(), "Run your base PDF loader first to define 'pages_and_texts'."
chapter_chunks = chapter_chunk_pdf_pages(pages_and_texts)   # >> same result as structure_chunked_pages
...
show_random_chapter_chunks(chapter_chunks, k=5, seed=21)

Showing 5 random chapters out of 171 total:

#1
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║ Chapter 42  |  Image by John Towner on unsplash.co m / CC0 Lifestyles and Nutrition  | p21–29  |  ~tokens 3135.25║
╟──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╢
║ Image by  John Towner  on  unsplash.co m / CC0  Lifestyles and Nutrition  UNIVERSITY OF HAWAI‘I                  ║
║ AT MĀNOA FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM AND HUMAN NUTRITION PROGRAM  In addition to                   ║
║ nutrition, health is affected by genetics, the  environment, life cycle, and lifestyle. One                      ║
║ facet of lifestyle is your  dietary habits. Recall that we discussed briefly how nutrition                       ║
║ affects  health. A greater discussion of this will follow in subsequent  chapters in this book,                  ║
║ as there is an

In [48]:
for c in structure_chunked_pages[:10]:
    print(c["page_start"], "|", c["title"][:70])

-39 | Human Nutrition: 2020 Edition
-38 | Human Nutrition: 2020 Edition by
-37 | Contents Preface
-36 | Lifestyles and Nutrition
-35 | The Cardiovascular System
-34 | Indicators of Health: Body Mass Index, Body Fat Content, and Fat Distr
-33 | Chloride University of Hawai‘i at Mānoa Food Science and Human Nutriti
-32 | The Functions of Carbohydrates in the Body
-31 | Nonessential and Essential Fatty Acids
-30 | Defining Protein


In [50]:
# Build stats + plots directly from your existing chunk lists
# (chunked_pages, semantic_chunked_pages, recursive_chunked_pages,
#  structure_chunked_pages, llm_chunked_pages)
# >> All five lists from Steps 3.1 to 3.5 must exist, so run those cells first.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# >> matplotlib.pyplot is Python's standard charting library; "plt" is the usual nickname.

# ---- Config: choose which size metric to analyze ----
# one of: "chars" (chunk_char_count), "words" (chunk_word_count), "tokens" (chunk_token_count)
METRIC = "words"
# >> A setting at the top of the cell: change it once and every number and chart follows.
# >> CAPITALS is a convention meaning "a setting, not meant to change while running".

def _size_val(c, metric: str):
    # >> Returns the size of ONE chunk, measured the way METRIC asks.
    if metric == "chars":
        return c.get("chunk_char_count", len(c.get("chunk_text","")))
        # >> dict.get(key, default): the value if the key exists, otherwise the default.
        # >> Here: use the stored count, or count it from the text if it's missing.
        # >> Unlike c["key"], .get() never raises a KeyError.
    if metric == "words":
        return c.get("chunk_word_count", len(c.get("chunk_text","").split()))
    if metric == "tokens":
        # fall back to chars/4 if not present
        return c.get("chunk_token_count", len(c.get("chunk_text",""))/4)
    raise ValueError("METRIC must be one of {'chars','words','tokens'}")
    # >> RAISE: deliberately stop with an error if METRIC is misspelled,
    # >> instead of silently returning nothing.

def analyze_chunks(chunks: list[dict], method_name: str, metric: str) -> dict:
    sizes = [_size_val(c, metric) for c in chunks]
    # >> list comprehension: one size per chunk, e.g. [62, 80, 41, ...]
    return {
        "Method": method_name,
        "Avg Chunk Size": float(np.mean(sizes)) if sizes else 0.0,
        # >> np.mean = average. "if sizes else 0.0" avoids an error on an empty list.
        # >> float() turns NumPy's number type into a plain Python number.
        "Num Chunks": len(sizes),
        "Size Variance": float(np.var(sizes)) if sizes else 0.0,
        # >> VARIANCE = how spread out the sizes are (see below).
    }

# ---- Gather results from the previously computed lists ----
datasets = [
    ("fixed",     chunked_pages),
    ("semantic",  semantic_chunked_pages),
    ("recursive", recursive_chunked_pages),
    ("structure", structure_chunked_pages),
    # ("llm",       llm_chunked_pages),
]
# >> A LIST OF TUPLES: each tuple pairs a label with its chunk list.
# >> A tuple ( , ) is like a list but fixed: it can't be changed after creation.

results = [analyze_chunks(chks, name, METRIC) for name, chks in datasets]
# >> TUPLE UNPACKING inside a list comprehension: each tuple splits into name and chks.
# >> Result: a list of 5 dictionaries, one per strategy.
df = pd.DataFrame(results)
# >> list of dictionaries -> table (each dictionary becomes a row), as in Step 2
print(df.round(3).to_string(index=False))
# >> .round(3) = 3 decimal places; .to_string(index=False) = print without the 0-4 row numbers

# --- Performance Analysis ---
print("\n# Performance analysis")
# >> "\n" at the start prints a blank line first
print("1. Structure-based chunking produced the most coherent sections.")
print("2. Semantic chunking maintained best context preservation.")
print("3. Fixed-size chunking showed lowest variance but poor semantic coherence.")
print("4. Recursive chunking provided balanced results.")

# >> These five lines are FIXED TEXT, not calculated. They print the same words whatever
# >> your numbers are. They're his conclusions, not something the code measured.

   Method  Avg Chunk Size  Num Chunks  Size Variance
    fixed          62.552        3321        583.291
 semantic          17.181       12092        241.871
recursive          85.387        2433       1476.866
structure        1214.801         171    1712584.803

# Performance analysis
1. Structure-based chunking produced the most coherent sections.
2. Semantic chunking maintained best context preservation.
3. Fixed-size chunking showed lowest variance but poor semantic coherence.
4. Recursive chunking provided balanced results.
